In [ ]:
import tkinter as tk
from tkinter import *
import json
import winsound 
class SpaceInvaders(object):
    def __init__(self):
        self.root = tk.Tk()
        self.root.title("Space Invaders")
#       responsible for grouping and organazing other widgets
        self.frame = tk.Frame(self.root)
        self.frame.pack(side="top", fill="both")
        self.game = Game(self.frame)

    def play(self):
        self.game.start_animation()
        self.root.bind("<Key>",self.game.keypress) # deplacements et tire du defender 
        self.root.mainloop()

: 

In [ ]:
class Game(object):
    def __init__(self, frame):
        self.width=600
        self.height=600
        self.frame = frame
        self.move_delta = 20 
        self.canvas = tk.Canvas(self.frame, width=self.width, height=self.height,bg="black")
        self.fleet = Fleet()
        self.defender = Defender() 
        self.defender.install_in(self.canvas)
        self.fleet.install_in(self.canvas)
        self.canvas.pack()        
#       controle le mouvement et le tire des bullets
    def keypress(self, event):
#         affecting defender coords to check its position
#         x is the value with which we move the defender
            x1,y1,x2,y2  = self.canvas.coords(self.defender.id)
            x=0
            if event.keysym == 'Left':
                if x1>0:
                    x=-10
            elif event.keysym == 'Right':
                if x2<600:
                    x=10
# une fois la touche espace est pressée, la fonction fire est executée
            elif event.keysym == 'space':
                self.defender.fire(self.canvas)  
                
            self.defender.move_in(self.canvas,x)

   #appel de toutes les fonctions a animer       
    def animation(self):
         if len(self.fleet.aliens_fleet)!=0: 
            x0,y0,x1,y1 = self.canvas.bbox("alien")
            if y1<(self.height-self.defender.height):
                self.move_bullets()
                self.move_aliens_fleet()
                self.fleet.manage_touched_aliens_by(self.canvas, self.defender)
                self.canvas.after(100, self.animation)
            else: 
                winsound.PlaySound('mixkit-arcade-retro-game-over-213', winsound.SND_FILENAME) 
                self.canvas.create_text(self.width//2,self.height//2,fill="red",font="Time 30 italic",text=" Vous avez perdu!")
         else:
            winsound.PlaySound('mixkit-arcade-retro-game-over-213', winsound.SND_FILENAME) 
            self.canvas.create_text(self.width//2,self.height//2,fill="red",font="Time 30 italic",text=" Vous avez gangé!")    
     
           
  # appliquer l'animation 
    def start_animation(self):
        self.canvas.after(10, self.animation)  
  #permet de appliquer la fonction move_in sur toutes les bullets tirées
    def move_bullets(self):
        for i in self.defender.fired_bullets:
            i.move_in(self.canvas)
            
    def move_aliens_fleet(self):
        self.fleet.move_in(self.canvas)

: 

In [ ]:
class Alien(object):
    image_alien = None
    def __init__(self):
        self.id = None
        self.alive = True
#      importer l'image alien.gif        
    @classmethod
    def get_image(cls):
        if cls.image_alien==None:
            cls.image_alien = tk.PhotoImage(file='alien.gif')
        return cls.image_alien
#permet de faire l'explosion   
    def touched_by(self, canvas, projectile):
#         import l'image
        self.pim = tk.PhotoImage(file='explosion.gif')
        xa1,ya1,xa2,ya2=canvas.bbox(projectile)
        explosion = canvas.create_image(xa1+(xa2-xa1), ya1+(ya2-ya1), image=self.pim, tags="explosion")
        canvas.after(100,canvas.delete,explosion)
#         permet la creation et l'affichage de l'image
#         x et y sont les coords de l'installation de l'image
#         image = image apportée en utilisant la fonction get_image
#         tags est la liste des groupes auxguels appartient les elements
    def install_in(self, canvas, x, y, image, tag):
        self.id = canvas.create_image(x, y, image=self.get_image(), tags='alien')

    def move_in(self, canvas, dx, dy):
        canvas.move(self.id, dx, dy)


: 

In [ ]:
class Fleet(object):
    def __init__(self):
        self.aliens_lines = 4
        self.aliens_columns = 5
#        la distance entre les images d'aliens
        self.aliens_inner_gap = 20 
#       les distances dont les quelle on deplace l'alien 
        self.alien_x_delta = 5
        self.alien_y_delta = 15
#        nombre des aliens
        fleet_size = self.aliens_lines*self.aliens_columns
#     liste des aliens
        self.aliens_fleet = [None] * fleet_size

    def get_width(self):
        return 700
    def get_height(self):
        return 600

     
    def install_in(self, canvas):
        #     la position initiale du premier alien
        x, y = 50,50 
        self.alien=Alien()
#         on parcours tous les aliens dans la liste aliens_fleet
        for i in range(0,len(self.aliens_fleet)):
#         on installe l'alien
            self.aliens_fleet[i]=Alien()
            self.aliens_fleet[i].install_in(canvas,x,y,self.alien.get_image(),'alien')
#         si on est encore loins de la fin de la fenetre on ajoute a la valeur de x la largeur de 
#         l'alien(73) plus la distance entre les aliens
            if (x+73+self.aliens_inner_gap) < int(canvas.cget('width')):
                x = x+self.aliens_inner_gap+73  
#         si on est arrivé a la fin on revient a la position initiale (x=50) et on retourne a 
#         la ligne on ajoutant la hauteur de l'image de l'alien(y=y+53) 
            elif (y+53+self.aliens_inner_gap) < int(canvas.cget('height')): 
                x=50
                y= y+self.aliens_inner_gap+53
#        deplacements du fleet
    def move_in(self, canvas):
        dx,dy = 0,0
        if len(self.aliens_fleet) != 0:
#         on prend les coords de tout les aliens
            x1,y1,x2,y2 = canvas.bbox("alien")
#      on retourne a la ligne (dy)
#      si fleet est arrivé a la fin de la fenetre on recule par -5
        if x2>=int(canvas.cget("width")): 
            self.alien_x_delta=-5
            dy = self.alien_y_delta
#     sinon si on est au debut de la fenetre on avance par 5
        elif x1<=0: 
            self.alien_x_delta=5
            dy = self.alien_y_delta
        else: #milieu
            dy = 0
# appliquer les conditions de deplacement pour chaque alien
#utilisation de la fonction move_in de la calsse alien 
        for i in range (len(self.aliens_fleet)):
            self.aliens_fleet[i].move_in(canvas,self.alien_x_delta,dy)
    def manage_touched_aliens_by(self,canvas,defender):
#       on prend la largeur et longueur de l'image
        dx = self.alien.get_image().width()
        dy = self.alien.get_image().height()
#       on parcours la liste des aliens
        for alien in self.aliens_fleet:
            x1, y1 = canvas.coords(alien.id)
            x2 = x1 + dx
            y2 = y1 + dy
            ids = canvas.find_overlapping(x1-37, y1-37, x2-37, y2-37)

            for bullet in defender.fired_bullets:
                if bullet.id in ids:
#         pour l'explosion
                    self.alien.touched_by(canvas, bullet.id)
#         on supprime l'alien et bullet dans le canvas
                    canvas.delete(bullet.id)
                    canvas.delete(alien.id)
#         on supprime la bullet de la liste fired_bullets
                    defender.fired_bullets.remove(bullet)
#     on retire l'alien de la liste de fleet
                    self.aliens_fleet.remove(alien)

: 

In [ ]:
class Defender(object):
    def __init__(self): 
        self.width = 20
        self.height = 20
        self.move_delta = 20 
        self.id = None
        self.max_fired_bullets = 8 
        self.fired_bullets = []   

    def get_id(self):
        return self.id
#     function called in Game class
#     canvas refers to the Canvas
#     creation d'un caree
    def install_in(self, canvas):
        self.canvas=canvas
        x0=(int(canvas.cget('width')) // 2)-(self.width // 2)
        y0=int(canvas.cget('height'))-self.height
        x1=x0 + self.width
        y1=y0+ self.height
        self.id=canvas.create_rectangle(x0,y0,x1,y1, fill='green')
#         function called in keypress function in game class
#         canvas refers to the Canvas
#         la fonction permet de deplacer le defender
#         pour les arguments: self.id= l'objet a deplacer, dx=la valeur dont la quelle on deplace 
#         le defender, la valeur 0 pour que le defender reste fixe sur l'axe des ordonnées           
        
    def move_in(self,canvas, dx): 
        canvas.move(self.id, dx, 0)
#         avant l'animation cette fonction install et affiche self.max_fired_bullets
#         a chaque installation d'une boullet on l'ajoute a la liste self.fired_bullets
#         si on depasse le nombre des bullets autoriséeson arrete le tire des bullets
    def fire(self, canvas):
         if len(self.fired_bullets) < self.max_fired_bullets: 
            bullet = Bullet(self)
            bullet.install_in(canvas)
            self.fired_bullets.append(bullet)
    

: 

In [ ]:
class Bullet(object):
    def __init__(self, shooter):
        self.radius = 5
        self.color = "red"
        self.speed = -18
        self.id = None
        self.shooter = shooter
#     canvas.coords(self.shooter.id) prend les coords du defender
#     creation d'une boullet
    def install_in(self, canvas):

        x1,y1,x2,y2 = canvas.coords(self.shooter.id)
        x = x1 + self.shooter.width//2 -self.radius
        y = y1 +self.radius
        x0=x2 -2*self.radius
        y0= y1
        self.id = canvas.create_oval(x, y, x2, y2, fill=self.color)
#      meme fonctionalité aue Defender().move_in()

    def move_in(self, canvas):
        canvas.move(self.id,0,self.speed)

: 

In [ ]:
SpaceInvaders().play() 

: 